# Programming for Social Robots

張閎鈞 (r13227136@ntu.edu.tw)

NTU Social Lab 內部培訓工作坊教材。  
適合沒有或僅有少量程式經驗的學習者，從零開始學習社交機器人應用開發。

---

## 本節目標

在這個 Notebook 中，你將學會：

1. 安裝必要套件並設定開發環境
2. 申請 OpenAI API 金鑰並安全地儲存
3. 呼叫 OpenAI API 取得 AI 回應
4. 理解 API 回應的資料結構
5. 建立一個具有對話記憶的 Chatbot
6. 透過 System Prompt 自訂 AI 角色

---

## 開始前的確認清單

> 請確認以下步驟都已完成：
> 1. 已安裝 Python 與 Conda（參考 `README.md`）
> 2. 已啟用 `socialrobot` 環境：在終端機執行 `conda activate socialrobot`
> 3. 在 VSCode 右上角選擇正確的 Kernel（Select Kernel → `socialrobot`）

In [ ]:
# 測試 Python 環境是否正常
print("Hello, Social Robots!")
print("Python 環境設定成功！")

---

## 什麼是 API？什麼是 LLM？

### LLM（大型語言模型）

LLM 是 **Large Language Model** 的縮寫，例如 OpenAI 的 **GPT-4o**、Google 的 **Gemini**。  
它們是經過大量文字資料訓練的 AI 模型，能夠理解並生成自然語言。

### API（應用程式介面）

API 是 **Application Programming Interface** 的縮寫。  
你可以把它想像成**餐廳的點餐服務**：

| 角色 | 餐廳比喻 | 程式世界 |
|------|---------|----------|
| 你 | 顧客 | 你的程式 |
| 菜單 | 可點的餐點 | API 文件 |
| 服務生 | 傳遞訂單 | API 請求 |
| 廚房 | 料理食物 | OpenAI 伺服器（LLM）|
| 餐點 | 上菜 | API 回應 |

### 為什麼要用 API？

自己在電腦上跑一個 LLM 需要超強的硬體（數十 GB 的 GPU 記憶體）。  
透過 API，你只需要網路連線，就能使用 OpenAI 伺服器上強大的模型。

---

## 步驟 1：安裝必要套件

In [ ]:
# 安裝 openai 套件與 python-dotenv（用來讀取 .env 檔案中的 API 金鑰）
!pip install openai python-dotenv

---

## 步驟 2：申請 API 金鑰並建立 `.env` 檔案

### 申請 OpenAI API 金鑰

1. 前往 [OpenAI Platform](https://platform.openai.com) 登入或註冊帳號
2. 點選右上角頭像 → **API Keys**
3. 點擊 **Create new secret key**，設定名稱後複製金鑰

> ⚠️ 金鑰只會顯示一次，請立刻複製並妥善保存！

### 建立 `.env` 檔案

在 `00_simple_chatbot/` 資料夾中建立一個名為 `.env` 的檔案，填入你的金鑰：

```
OPENAI_API_KEY=你的金鑰貼在這裡
```

### 為什麼要用 `.env`？

API 金鑰就像密碼，不能直接寫在程式碼裡，原因：
- 程式碼可能上傳到 GitHub，任何人都能看到
- 金鑰外洩會被盜用，產生費用
- 之後的範例也需要自己創建 .env file

`.env` 檔案已加入 `.gitignore`，不會被上傳到 GitHub，安全地存放機密資訊。

In [ ]:
import os
from dotenv import load_dotenv

# 載入 .env 檔案中的環境變數
load_dotenv()

# 確認 API 金鑰已正確載入（只顯示前幾個字元，避免外洩）
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"✅ API 金鑰載入成功：{api_key[:8]}...")
else:
    print("❌ 找不到 API 金鑰，請確認 .env 檔案是否正確設定")

---

## 步驟 3：第一次 API 呼叫

一個最基本的 API 請求需要兩個關鍵參數：

- **`model`**：要使用哪個模型。這裡使用 `gpt-4o-mini`，速度快且費用低，適合開發測試
- **`messages`**：對話內容，是一個「訊息陣列」

每則訊息有兩個欄位：

| 欄位 | 說明 | 可填入的值 |
|------|------|----------|
| `role` | 誰說的話 | `"user"`（使用者）、`"assistant"`（AI）、`"system"`（系統設定）|
| `content` | 訊息的文字內容 | 任何字串 |

In [ ]:
from openai import OpenAI

# 建立 OpenAI 客戶端，帶入 API 金鑰
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 發送請求
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "你好！請用一句話介紹你自己。"},
    ],
)

# 取得 AI 的回覆文字
reply = response.choices[0].message.content
print("AI 回覆：", reply)

### 認識回應物件（Response Object）

API 回傳的不只是文字，而是一個包含許多資訊的物件。  
讓我們來探索它的結構，特別是 **token 用量**（影響費用的關鍵）：

In [ ]:
# 查看常用欄位
print("=== 回應資訊 ===")
print(f"使用的模型：{response.model}")
print(f"完成原因：{response.choices[0].finish_reason}")
print()
print("=== Token 用量（影響費用）===")
print(f"  輸入（prompt）token 數：{response.usage.prompt_tokens}")
print(f"  輸出（completion）token 數：{response.usage.completion_tokens}")
print(f"  總計 token 數：{response.usage.total_tokens}")
print()
print("=== 完整回應物件（進階參考）===")
print(response)

---

## 步驟 4：建立有記憶的 Chatbot

目前的程式每次只傳送單一訊息，AI 並不記得之前的對話。  
要讓 AI「記住」對話，我們需要每次都把**完整的對話歷史**一起傳送。

```
第 1 輪：messages = [user: "你好"]
第 2 輪：messages = [user: "你好", assistant: "你好！有什麼可以幫你？", user: "你叫什麼名字？"]
第 3 輪：messages = [... 前兩輪 ..., assistant: "我是 GPT-4o-mini", user: "你能做什麼？"]
```

每一輪都把整個對話歷史帶上，AI 就能理解上下文。

In [ ]:
# 對話歷史（從空的開始）
conversation_history = []

print("=== 簡易 Chatbot ===")
print("輸入『再見』結束對話\n")

while True:
    # 取得使用者輸入
    user_input = input("你：")

    # 結束條件
    if "再見" in user_input:
        print("AI：再見！很高興和你聊天。")
        break

    # 把使用者訊息加入歷史
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    # 呼叫 API（帶上完整對話歷史）
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=conversation_history,
    )

    # 取得 AI 回覆文字
    ai_reply = response.choices[0].message.content

    # 把 AI 回覆也加入歷史（下一輪要帶上）
    conversation_history.append({
        "role": "assistant",
        "content": ai_reply
    })

    print(f"AI：{ai_reply}\n")

---

## 步驟 5：自訂角色 — System Prompt

透過 `system` 角色的訊息，可以設定 AI 的**個性、背景、能力限制**，讓它扮演特定角色。

System Prompt 通常放在對話歷史的**最前面**，只需設定一次，整個對話都有效。

例如，以下的 System Prompt 讓 AI 扮演一個社交機器人「Pepper」：

In [ ]:
# 設定 System Prompt（定義 AI 的角色與行為規則）
SYSTEM_PROMPT = """你是一個名叫「Pepper」的社交機器人助理，在 NTU 社交實驗室服務。

你的個性：
- 親切、有耐心、充滿活力
- 說話簡短清楚，適合與人面對面互動
- 每次回覆不超過 2 句話
- 當使用者說再見時，溫暖地回應並結束對話
"""

# 對話歷史（預先放入 system prompt）
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

print("=== Pepper 機器人 ===")
print("輸入『再見』結束對話\n")

while True:
    user_input = input("你：")

    # 把使用者訊息加入歷史
    conversation_history.append({
        "role": "user",
        "content": user_input
    })

    # 呼叫 API
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=conversation_history,
    )

    ai_reply = response.choices[0].message.content

    # 把 AI 回覆加入歷史
    conversation_history.append({
        "role": "assistant",
        "content": ai_reply
    })

    print(f"Pepper：{ai_reply}\n")

    # 若 AI 的回覆表示結束對話則停止（讓 AI 決定何時結束）
    if "再見" in user_input:
        break

---

## 恭喜完成！🎉

你已經學會：

| 技能 | 對應程式碼 |
|------|----------|
| 安全儲存 API 金鑰 | `.env` + `load_dotenv()` |
| 呼叫 LLM API | `client.chat.completions.create()` |
| 讀取 AI 回覆 | `response.choices[0].message.content` |
| 建立對話記憶 | `conversation_history` 陣列 |
| 自訂 AI 角色 | `{"role": "system", "content": ...}` |

### 下一步

接下來的章節中，我們將把這個 Chatbot 包裝成一個 **HTTP API 伺服器**（使用 FastAPI），讓其他程式、App 或機器人都能呼叫它！

→ 前往 `01_fastapi_chat/`